In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns
import os
import json 

In [ ]:
sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (12, 6)

#Loading the data
df = pd.read_csv('/Users/tahminasahar/earthquake-risk-afghanistan/data/usgs_afghanistan_raw.csv')
df['time'] = pd.to_datetime(df['time'])
df['year'] = df['time'].dt.year
df['month'] = df['time'].dt.month

print(f" Loaded {len(df)} earthquakes")
print(f"Columns: {df.columns.tolist()}")

 Loaded 8250 earthquakes
Columns: ['time', 'latitude', 'longitude', 'depth', 'mag', 'magType', 'nst', 'gap', 'dmin', 'rms', 'net', 'id', 'updated', 'place', 'type', 'horizontalError', 'depthError', 'magError', 'magNst', 'status', 'locationSource', 'magSource', 'year', 'month']


In [ ]:
# Feature 1: Depth Category
# Categorize each earthquake by how deep it was
depth_bins = [0, 70, 300, 800]
depth_labels = ['Shallow', 'Intermediate', 'Deep']
df['depth_category'] = pd.cut(df['depth'], bins=depth_bins, labels=depth_labels, right=False)

# Feature 2: Depth Risk Score
# Convert depth category into a NUMBER for the ML model
# ML models need numbers — they can't understand words like "Shallow"
depth_risk_map = {
    'Shallow': 3,       # Shallow earthquakes are more dangerous
    'Intermediate': 2,  # Intermediate depth is less dangerous
    'Deep': 1          # Deep earthquakes are least dangerous
}

df['depth_risk_score'] = df['depth_category'].map(depth_risk_map)

print("Depth categories created:")
print(df['depth_category'].value_counts())
print(f"\nDepth risk score sample:")
print(df[['depth', 'depth_category', 'depth_risk_score']].head(10))

Depth categories created:
depth_category
Intermediate    4764
Shallow         3479
Deep               7
Name: count, dtype: int64

Depth risk score sample:
     depth depth_category depth_risk_score
0  224.419   Intermediate                2
1  114.000   Intermediate                2
2  223.091   Intermediate                2
3  143.537   Intermediate                2
4   10.000        Shallow                3
5   64.259        Shallow                3
6  102.823   Intermediate                2
7  178.480   Intermediate                2
8  113.421   Intermediate                2
9  209.319   Intermediate                2


In [ ]:
#Feature 3
mag_bins = [4.0, 4.9, 5.9, 6.9, 10.0]
mag_labels = ['Light', 'Moderate', 'Strong', 'Major']

df['mag_category'] = pd.cut(
    df['mag'],
    bins=mag_bins,
    labels=mag_labels,
    right=False
)

#  FEATURE 4: Magnitude Risk Score 
mag_risk_map = {
    'Light':    1,
    'Moderate': 2,
    'Strong':   3,
    'Major':    4
}
df['mag_risk_score'] = df['mag_category'].map(mag_risk_map)

#FEATURE 5: Energy Proxy 
df['energy_proxy'] = 10 ** (1.5 * df['mag'])

print("✅ Magnitude features created!")
print("\nEarthquakes per magnitude category:")
print(df['mag_category'].value_counts())
print("\nSee the energy difference — notice how huge it gets:")
print(df[['mag', 'mag_category', 'energy_proxy']].head(8))


In [ ]:
#Distance From Kabul 
KABUL_LAT = 34.5553
KABUL_LON = 69.2075

# Haversine formula — real distance on a curved Earth
def haversine_distance(lat1, lon1, lat2, lon2):
    
    # Step 1: Convert degrees to radians
    # Why? Math functions (sin, cos) need radians not degrees
    # Degrees are for humans, radians are for math
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)
    
    # Step 2: Calculate the differences
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    # Step 3: Apply the Haversine formula
    # This accounts for Earth's spherical shape
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    # Step 4: Multiply by Earth's radius to get km
    earth_radius_km = 6371
    return c * earth_radius_km

# Apply to every earthquake row
df['dist_to_kabul'] = haversine_distance(
    df['latitude'],   # each earthquake's latitude
    df['longitude'],  # each earthquake's longitude
    KABUL_LAT,        # Kabul's latitude
    KABUL_LON         # Kabul's longitude
)